# HW: Imbalanced data, Evaluation metrics, Cross validation

繳交網址：https://forms.gle/gjfsbxxtqrSKbuVV7 <br>
繳交期限：4/11 23:55 <br>
重複繳交以最新的做計算

HW-1

作業說明：請自行定義一個函式 def split_data(): 切分imbalanced data<br>
情境：一份DF有N個類別，放入split_data()之後自動切出x_train, y_train, x_test, y_test，<br>
其中x_train, y_train, x_test, y_test內比例與DF中各類別比例需相同。<br>
函式input：自行定義<br>
函式output：x_train, y_train, x_test, y_test<br>
不可使用from sklearn.model_selection import train_test_split套件，其他皆可使用<br>
<br>
配分方式(總分60)：<br>
1. 按label比例分(得60分)，未按label比例(得40分)

Hint: 可參考Split data to train and test那段<br>
1. 整理不平衡資料時，需要將不同類別的資料單獨拉出來，以方便後續切分資料所使用
2. 創建temp_df存取不同類別的資料，創建out_df存取最終return的資料
3. out_df = pd.concat([out_df, temp_df], axis=0)
3. 檢查是否與raw data的分配比例一樣

In [56]:
import pandas as pd
import random as rd
import math
input_df = pd.read_csv('sample_data/HW1_data.csv')

# 新增區段

In [57]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
#題目不是很懂,我的作法為計算出所有class(10組)之比例,並將data照比例且平均分為x,y兩組
def split_data(input_df, test_size):
    #總需數量
    num_data = len(input_df)
    train_len = int(num_data * (1-test_size))
    test_len = num_data - train_len
    #以y欄位分組, 計算每組所需的train及test數量
    labels = input_df['y']
    unique_labels = set(labels)
    total = len(input_df)
    labels_ratio = []
    trainNum = []
    testNum = []
    for i in unique_labels:
      temp = sum(input_df['y']== i)
      labels_ratio.append(temp / total)
      trainNum.append(math.floor(temp / total * (1-test_size) * total))
      testNum.append(math.floor(temp / total * test_size * total))
    #資料分組
    countClass = 0
    countData = 0
    x_train = pd.DataFrame(columns=['Sequence_Name','mcg','gvh','alm','mit','erl','pox','vac','nuc','y'])
    y_train = pd.DataFrame(columns=['Sequence_Name','mcg','gvh','alm','mit','erl','pox','vac','nuc','y'])
    x_test = pd.DataFrame(columns=['Sequence_Name','mcg','gvh','alm','mit','erl','pox','vac','nuc','y'])
    y_test = pd.DataFrame(columns=['Sequence_Name','mcg','gvh','alm','mit','erl','pox','vac','nuc','y'])
    roundData = pd.DataFrame(columns=['Sequence_Name','mcg','gvh','alm','mit','erl','pox','vac','nuc','y'])
    for i in unique_labels:
      tempData = input_df[input_df['y'] == i]
      needXTrainNum = math.floor(trainNum[countClass] / 2)
      needYTrainNum = trainNum[countClass] - needXTrainNum
      countXNum = 0
      countYNum = 0
      #依照labels_train,取出train組
      for index, row in tempData.iterrows():
        rnd = rd.random()
        if (rnd <= 0.5) and (countXNum < needXTrainNum):
          x_train = x_train.append(row, ignore_index = True)
          tempData = tempData.drop([index])
          countXNum = countXNum + 1
        elif (rnd > 0.5 ) and (countYNum < needYTrainNum):
          y_train = y_train.append(row, ignore_index = True)
          tempData = tempData.drop([index])
          countYNum = countYNum + 1
      #依照labels_test,取出test組
      needXTestNum = math.floor(testNum[countClass] / 2)
      needYTestNum = testNum[countClass] - needXTestNum
      countXNum = 0
      countYNum = 0
      for index, row in tempData.iterrows():
        rnd = rd.random()
        if (rnd <= 0.5) and (countXNum < needXTestNum):
          x_test = x_test.append(row, ignore_index = True)
          tempData = tempData.drop([index])
          countXNum = countXNum + 1
        elif (rnd > 0.5 ) and (countYNum < needYTestNum):
          y_test = y_test.append(row, ignore_index = True)
          tempData = tempData.drop([index])
          countYNum = countYNum + 1
      for index, row in tempData.iterrows():
        roundData = roundData.append(row, ignore_index = True)
      countClass = countClass + 1
    #補齊無條件捨去的部分(如不補會遺失少數資料, 補了會稍微影響x組內資料比例)
    for index, row in roundData.iterrows():
      if len(x_train) + len(y_train) < train_len:
        x_train = x_train.append(row, ignore_index = True)
      else:
        x_test = x_test.append(row, ignore_index = True)
    return x_train, y_train, x_test, y_test

In [54]:
x_train, y_train, x_test, y_test = split_data(input_df, 0.2)
print(x_train)
print(y_train)
print(x_test)
print(y_test)

    Sequence_Name   mcg   gvh   alm   mit  erl  pox   vac   nuc    y
0      MAN1_YEAST  0.37  0.36  0.56  0.18  0.5  0.0  0.48  0.26  VAC
1      APE3_YEAST  0.74  0.57  0.51  0.28  0.5  0.0  0.58  0.22  VAC
2      CBPS_YEAST  0.70  0.65  0.54  0.28  0.5  0.0  0.60  0.40  VAC
3      AMPL_YEAST  0.42  0.32  0.48  0.16  0.5  0.0  0.55  0.22  VAC
4      PEP3_YEAST  0.51  0.46  0.49  0.21  0.5  0.0  0.55  0.22  VAC
..            ...   ...   ...   ...   ...  ...  ...   ...   ...  ...
590    SPR1_YEAST  0.66  0.65  0.54  0.40  0.5  0.0  0.51  0.22  EXC
591    VATF_YEAST  0.65  0.58  0.53  0.21  0.5  0.0  0.51  0.22  VAC
592    VPH1_YEAST  0.57  0.54  0.32  0.15  0.5  0.0  0.52  0.25  VAC
593    VP34_YEAST  0.57  0.47  0.48  0.17  0.5  0.0  0.52  0.26  VAC
594    RL3P_YEAST  0.24  0.25  0.57  0.28  0.5  0.0  0.51  0.11  CYT

[595 rows x 10 columns]
    Sequence_Name   mcg   gvh   alm   mit  erl  pox   vac   nuc    y
0      DAP2_YEAST  0.36  0.68  0.38  0.08  0.5  0.0  0.53  0.25  VAC
1      PE

HW-2

作業說明(總分40)： <br>
以def定義N類別的recall function (得20分)<br>
函式input：y_true, y_pred<br>
函式output：recall<br>
以def定義N類別的precision function (得20分)<br>
函式input：y_true, y_pred<br>
函式output：recall<br>
套件皆可使用

reference<br>
https://medium.com/data-science-in-your-pocket/calculating-precision-recall-for-multi-class-classification-9055931ee229

Hint: 可參考Evaluation metrics (HW-2)那段<br>
可以用confusing matrix幫忙計算<br>
可利用from sklearn.metrics import recall_score, precision_score<br>

recall:    array([0.6847, 0.6   , 0.7714, 0.4545, 0.7255, 0.8405, 0.9098, 0.9301,   0.85  , 0.9   ])<br>
precision: array([0.883 , 1.    , 0.7105, 0.6061, 0.6066, 0.7249, 0.8506, 0.8244,   0.68  , 0.871 ])

In [55]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
df=pd.read_csv('sample_data/HW2_data.csv')
y_true = df['y_true']
y_pred = df['y_pred']
le = LabelEncoder()
le.fit(y_true)
y_true = le.transform(y_true)
y_pred = le.transform(y_pred)
tmpRecall = multi_recall(y_true,y_pred)
tmpPrecision = multi_precision(y_true,y_pred)
print(tmpRecall)
print(tmpPrecision)

[0.6846652267818575, 0.6, 0.7714285714285715, 0.45454545454545453, 0.7254901960784313, 0.8404907975460123, 0.9098360655737705, 0.9300699300699301, 0.85, 0.9]
[0.883008356545961, 1.0, 0.7105263157894737, 0.6060606060606061, 0.6065573770491803, 0.7248677248677249, 0.8505747126436781, 0.8243801652892562, 0.68, 0.8709677419354839]


In [59]:
def multi_recall(y_true, y_pred):
  multi_recall = []
  unique_labels = set(y_true)
  total = len(y_true)
  for positiveLabel in unique_labels:
    TP = 0
    FP = 0
    FN = 0
    for i in range(total):
      if (positiveLabel == y_true[i]) and (y_true[i] == y_pred[i]):
        TP = TP + 1
      elif (positiveLabel != y_true[i]) and (positiveLabel == y_pred[i]):
        FP = FP + 1
      elif (positiveLabel == y_true[i]) and (positiveLabel != y_pred[i]):
        FN = FN + 1
    multi_recall.append(TP / (TP + FN))
  return multi_recall
  
multi_recall(df["y_true"], df["y_pred"])

[0.9,
 0.6,
 0.85,
 0.6846652267818575,
 0.8404907975460123,
 0.9098360655737705,
 0.7254901960784313,
 0.45454545454545453,
 0.9300699300699301,
 0.7714285714285715]

In [60]:
def multi_precision(y_true, y_pred):
  multi_precision = []
  unique_labels = set(y_true)
  total = len(y_true)
  for positiveLabel in unique_labels:
    TP = 0
    FP = 0
    FN = 0
    for i in range(total):
      if (positiveLabel == y_true[i]) and (y_true[i] == y_pred[i]):
        TP = TP + 1
      elif (positiveLabel != y_true[i]) and (positiveLabel == y_pred[i]):
        FP = FP + 1
      elif (positiveLabel == y_true[i]) and (positiveLabel != y_pred[i]):
        FN = FN + 1
    multi_precision.append(TP / (TP + FP))
  return multi_precision

multi_precision(df["y_true"], df["y_pred"])

[0.8709677419354839,
 1.0,
 0.68,
 0.883008356545961,
 0.7248677248677249,
 0.8505747126436781,
 0.6065573770491803,
 0.6060606060606061,
 0.8243801652892562,
 0.7105263157894737]